In [1]:
import coiled

import fsspec
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import dask
import sparse
from dask.distributed import Client, LocalCluster
from dask.distributed import print

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [4]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=2,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r5.2xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r5.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   flox         │ Wheel built from ~/flox-0.10.3.tar.gz                       │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [ ]:
model_version = "version_0_3_2"
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
run_date = "20250507"
interval = "2015_2016"

In [17]:
gross_emis_CO2_only_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif"], 
                                         name = 'emis_all_C_pools_CO2_only')
gross_emis_all_gases_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif"], 
                                         name = 'emis_all_C_pools_all_gases')
node_tile_uri = pd.Series([f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__land_state_node_{interval}.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__land_state_node_{interval}.tif"], 
                          name = 'node_codes')
print(gross_emis_CO2_only_tile_uri)
print(gross_emis_all_gases_tile_uri)
print(node_tile_uri)
print(type(gross_emis_CO2_only_tile_uri))

0    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
1    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
Name: emis_all_C_pools_CO2_only, dtype: object
0    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
1    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
Name: emis_all_C_pools_all_gases, dtype: object
0    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
1    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
Name: node_codes, dtype: object
<class 'pandas.core.series.Series'>


In [ ]:
gross_emis_CO2_only_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/2016_2017/_pixel_yr/4000_pixels/20250507/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_2015_2016.tif"], name = 'emis_all_C_pools_CO2_only')
gross_emis_all_gases_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/2016_2017/_pixel_yr/4000_pixels/20250507/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_2015_2016.tif"], name = 'emis_all_C_pools_all_gases')
node_tile_uri = pd.Series(["s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_1/land_state_node/standard_model/annual_intervals/2015_2016/4000_pixels/20250507/00N_020E__23_-4_24_-3__land_state_node_2015_2016.tif"], name = 'node_codes')
print(gross_emis_CO2_only_tile_uri)
print(node_tile_uri)
print(type(emis_tile_uri))

In [8]:
import re

def parse_metadata_from_uri(uri: str):
    """
    Extracts year, chunk, and variable from an S3 URI string.
    """
    uri = uri.values.tolist()[0]
    print(uri)
    # This regex captures year, chunk, and variable in your filename format
    pattern = r'(\d{4}_\d{4}).*?__(\d+_-?\d+_\d+_-?\d+)__([a-zA-Z0-9_]+)(?:_pixel_yr)?_\1'
    match = re.search(pattern, uri)
    
    if match:
        interval = match.group(1)
        chunk_id = match.group(2)
        variable = match.group(3)
    else:
        interval, chunk_id, variable = None, None, None
    
    return interval, chunk_id, variable

In [9]:
interval, chunk_id, variable = parse_metadata_from_uri(gross_emis_CO2_only_tile_uri)
print(interval)
print(chunk_id)
print(variable)

s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_3_2/gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/2015_2016/_pixel_yr/4000_pixels/20250507/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_2015_2016.tif
2015_2016
23_-4_24_-3
gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr


In [20]:
gross_emis_CO2_only = xr.open_mfdataset(
    gross_emis_CO2_only_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

gross_emis_all_gases = xr.open_mfdataset(
    gross_emis_all_gases_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

nodes = xr.open_mfdataset(
    node_tile_uri.values.tolist(),
    parallel=True
    # chunks={'x': 1000, 'y':1000}
).squeeze().persist()

print(gross_emis_CO2_only)
print(gross_emis_all_gases)
nodes

<xarray.Dataset>
Dimensions:      (x: 4000, y: 8000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -4.999 -4.999 -5.0 -5.0
    spatial_ref  int64 0
Data variables:
    band_data    (y, x) float32 dask.array<chunksize=(400, 400), meta=np.ndarray>
<xarray.Dataset>
Dimensions:      (x: 4000, y: 8000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -4.999 -4.999 -5.0 -5.0
    spatial_ref  int64 0
Data variables:
    band_data    (y, x) float32 dask.array<chunksize=(400, 400), meta=np.ndarray>


<xarray.Dataset>
Dimensions:      (x: 4000, y: 8000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -4.999 -4.999 -5.0 -5.0
    spatial_ref  int64 0
Data variables:
    band_data    (y, x) float64 dask.array<chunksize=(400, 400), meta=np.ndarray>

In [22]:
gross_emis_CO2_only_sub, nodes_aligned = xr.align(gross_emis_CO2_only, nodes, join="inner")
print(gross_emis_CO2_only_sub)
nodes_aligned

<xarray.Dataset>
Dimensions:      (x: 4000, y: 8000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -4.999 -4.999 -5.0 -5.0
    spatial_ref  int64 0
Data variables:
    band_data    (y, x) float32 dask.array<chunksize=(400, 400), meta=np.ndarray>


<xarray.Dataset>
Dimensions:      (x: 4000, y: 8000)
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -4.999 -4.999 -5.0 -5.0
    spatial_ref  int64 0
Data variables:
    band_data    (y, x) float64 dask.array<chunksize=(400, 400), meta=np.ndarray>

In [23]:
node_data = nodes_aligned.band_data
node_data.name = 'state_node'
node_data

<xarray.DataArray 'state_node' (y: 8000, x: 4000)>
dask.array<getitem, shape=(8000, 4000), dtype=float64, chunksize=(400, 400), chunktype=numpy.ndarray>
Coordinates:
    band         int64 1
  * x            (x) float64 23.0 23.0 23.0 23.0 23.0 ... 24.0 24.0 24.0 24.0
  * y            (y) float64 -3.0 -3.0 -3.001 -3.001 ... -4.999 -4.999 -5.0 -5.0
    spatial_ref  int64 0
Attributes:
    AREA_OR_POINT:  Area

In [24]:
node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
                       2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
                       2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
                      dtype=np.uint32)

In [26]:
from flox import ReindexArrayType, ReindexStrategy

gross_emis_CO2__by_node = xarray_reduce(
    gross_emis_CO2_only_sub.band_data,
    node_data,
    func='sum',
    expected_groups=(node_codes),
    reindex=ReindexStrategy(
        blockwise=False, array_type=ReindexArrayType.SPARSE_COO
    ),
    fill_value=0
    
)
gross_emis_CO2__by_node

<xarray.DataArray 'band_data' (state_node: 25)>
dask.array<groupby_nansum, shape=(25,), dtype=float32, chunksize=(25,), chunktype=sparse.COO>
Coordinates:
    band         int64 1
    spatial_ref  int64 0
  * state_node   (state_node) uint32 2211100 2211200 2212110 ... 5210000 5220000
Attributes:
    AREA_OR_POINT:  Area

In [28]:
gross_emis_CO2_result = gross_emis_CO2__by_node.compute()

In [ ]:
gross_emis_CO2_result

In [30]:
sparse_data = gross_emis_CO2_result.data

# Step 3: Extract coordinates and values
dim_names = gross_emis_CO2_result.dims
indices = sparse_data.coords
values = sparse_data.data

# Step 4: Map dimension indices to coordinate values
coord_dict = {
    dim: gross_emis_CO2_result.coords[dim].values[indices[i]]
    for i, dim in enumerate(dim_names)
}
coord_dict["value"] = values

df = pd.DataFrame(coord_dict)
df["year"] = 2016
df["chunk"] = "23_-4_24_-3"
df["variable"] = "gross_emissions__all_C_pools__CO2_only__MgCO2"

In [31]:
df

,state_node,value,year,chunk,variable
0,2211100,1.016439e+04,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
1,2211200,2.618886e+04,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
2,2212110,7.108322e+06,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
3,2212120,3.203444e+07,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
4,2212210,2.334183e+03,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
5,2212220,5.861226e+05,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
6,2214100,2.013405e+03,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
7,2214200,1.530667e+03,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
8,2215200,3.309150e+02,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2
9,2221100,4.828078e+04,2016,23_-4_24_-3,gross_emissions__all_C_pools__CO2_only__MgCO2


In [ ]:
df.head()

In [ ]:
df[(df.state_node == 2212210)]